In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

In [0]:
RAW_PATH = "/Volumes/workspace/ingestion/project_data/raw"

BRONZE_TABLE = "workspace.bronze.orders"

print("Raw path:", RAW_PATH)
print("Bronze table:", BRONZE_TABLE)

In [0]:
raw_schema = T.StructType([
    T.StructField("order_id", T.StringType(), True),
    T.StructField("customer_id", T.StringType(), True),
    T.StructField("order_date", T.StringType(), True),
    T.StructField("product_id", T.StringType(), True),
    T.StructField("product_name", T.StringType(), True),
    T.StructField("category", T.StringType(), True),
    T.StructField("quantity", T.StringType(), True),
    T.StructField("unit_price", T.StringType(), True),
    T.StructField("discount", T.StringType(), True),
    T.StructField("country", T.StringType(), True),
    T.StructField("payment_method", T.StringType(), True),
    T.StructField("order_status", T.StringType(), True)
]
                          )

In [0]:
raw_df = (
    spark.read
    .schema(raw_schema)
    .option("header", True)
    .option("mode", "PERMISSIVE")
    .csv(f"{RAW_PATH}/orders_*")
)

In [0]:
display(raw_df.limit(20))

In [0]:
bronze_df = (
    raw_df

    .withColumn(
        "source_file",
        F.element_at(
            F.split(
                F.col("_metadata.file_path"),
                "/"
            ),
            -1
        )
    )

    .withColumn(
        "batch_id",
        F.regexp_extract(
            F.col("_metadata.file_path"),
            r"orders_(\d+)",
            1
        )
    )

    .withColumn(
        "batch_id",
        F.when(
            F.col("batch_id") == "",
            F.lit("UNKNOWN")
        ).otherwise(
            F.concat(
                F.lit("BATCH_"),
                F.col("batch_id")
            )
        )
    )

    .withColumn(
        "ingestion_timestamp",
        F.current_timestamp()
    )
)

In [0]:
display(
    bronze_df.select(
        "order_id",
        "batch_id",
        "source_file",
        "ingestion_timestamp"
    ).limit(20)
)

In [0]:
BRONZE_TABLE = "workspace.bronze.orders"

(
    bronze_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(BRONZE_TABLE)
)

print(f"Bronze table created: {BRONZE_TABLE}")

In [0]:
spark.sql("""
SELECT COUNT(*) AS total_records
FROM workspace.bronze.orders
""").show()

In [0]:
display(
    spark.sql("""
        SELECT *
        FROM workspace.bronze.orders
        LIMIT 20
    """)
)

In [0]:
display(
    spark.sql("""
        SELECT
            batch_id,
            COUNT(*) AS record_count
        FROM workspace.bronze.orders
        GROUP BY batch_id
        ORDER BY batch_id
    """)
)

Verify bad data was preserved

This is an important Bronze checkpoint.

In [0]:
spark.sql("""
SELECT COUNT(*) AS null_customer_count
FROM workspace.bronze.orders
WHERE customer_id IS NULL
""").show()

In [0]:
spark.sql("""
SELECT COUNT(*) AS invalid_quantity_count
FROM workspace.bronze.orders
WHERE TRY_CAST(quantity AS INT) <= 0
""").show()

In [0]:
display(
    spark.sql("""
        SELECT
            order_id,
            COUNT(*) AS record_count
        FROM workspace.bronze.orders
        GROUP BY order_id
        HAVING COUNT(*) > 1
        ORDER BY record_count DESC
    """)
)

In [0]:
spark.sql("""
DESCRIBE HISTORY workspace.bronze.orders
""").show(truncate=False)

In [0]:
spark.sql("""
DESCRIBE TABLE EXTENDED workspace.bronze.orders
""").show(truncate=False)

In [0]:
display(
    spark.sql("""
        SELECT
            COUNT(*) AS total_records,
            COUNT(DISTINCT order_id) AS unique_order_ids,
            COUNT(*) - COUNT(DISTINCT order_id) AS duplicate_order_ids,
            SUM(
                CASE
                    WHEN customer_id IS NULL THEN 1
                    ELSE 0
                END
            ) AS null_customer_ids,
            SUM(
                CASE
                    WHEN TRY_CAST(quantity AS INT) <= 0 THEN 1
                    ELSE 0
                END
            ) AS invalid_quantities,
            SUM(
                CASE
                    WHEN TRY_CAST(unit_price AS DOUBLE) < 0 THEN 1
                    ELSE 0
                END
            ) AS invalid_prices
        FROM workspace.bronze.orders
    """)
)